ARTI308 - Machine Learning

# Project Notebook 1: Exploratory Data Analysis (EDA)
## Heart Disease Prediction

**Goal of the project:** build a classification model that predicts whether a patient has heart disease (`HeartDisease` = 1) or not (`HeartDisease` = 0) from clinical measurements.

**Goal of this notebook:** understand the raw data *before* touching it. We do not change anything here; we only look.

### Dataset
Heart Failure Prediction Dataset (Kaggle, author: fedesoriano) - 918 patients, 11 input features and 1 target.
Check the license on the dataset's Kaggle page and cite it in the README.

| Column | Meaning |
|---|---|
| Age | age of the patient (years) |
| Sex | M / F |
| ChestPainType | TA, ATA, NAP, ASY |
| RestingBP | resting blood pressure (mm Hg) |
| Cholesterol | serum cholesterol (mg/dl) |
| FastingBS | 1 if fasting blood sugar > 120 mg/dl, else 0 |
| RestingECG | Normal, ST, LVH |
| MaxHR | maximum heart rate achieved |
| ExerciseAngina | exercise-induced angina (Y / N) |
| Oldpeak | ST depression induced by exercise |
| ST_Slope | slope of the peak exercise ST segment (Up, Flat, Down) |
| HeartDisease | **target**: 1 = heart disease, 0 = normal |

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
# Path relative to this notebook (notebooks/ -> data/raw/)
DATA_PATH = "../data/raw/heart.csv"

# On Google Colab you can read the raw file directly from GitHub instead:
# DATA_PATH = "https://raw.githubusercontent.com/<username>/Heart-Disease-Prediction/main/data/raw/heart.csv"

df = pd.read_csv(DATA_PATH)
df.head(7)

**Observation:**

*(Write what you notice after running the cell above.)*

## 3. Quick dataset checks
### 3.1 Shape

In [ ]:
print("Shape (rows, columns):", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

**Observation:**

*(Write what you notice after running the cell above.)*

### 3.2 Data types
Each column's type should match its real meaning (numbers as numbers, categories as text).

In [ ]:
df.dtypes

**Observation:**

*(Write what you notice after running the cell above.)*

### 3.3 Missing values (the obvious kind)
`isna()` only finds values that are literally empty (`NaN`).

In [ ]:
df.isna().sum()

**Observation:**

*(Write what you notice after running the cell above.)*

### 3.4 Duplicate rows

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

**Observation:**

*(Write what you notice after running the cell above.)*

### 3.5 Descriptive statistics
`include="all"` shows numerical and categorical columns together. Pay attention to the `min` row.

In [ ]:
df.describe(include="all")

**Observation:**

*(Write what you notice after running the cell above.)*

### 3.6 Hidden problems
`isna()` can say "no missing values" while the data still contains **impossible values** that were used as a placeholder.
A resting blood pressure of 0 or a cholesterol of 0 is not physically possible for a living patient.
We also check `Oldpeak` for negative values.

In [ ]:
print("Zeros in RestingBP:   ", (df["RestingBP"] == 0).sum())
print("Zeros in Cholesterol: ", (df["Cholesterol"] == 0).sum())
print("Negative Oldpeak:     ", (df["Oldpeak"] < 0).sum())

**Observation:**

*(Write what you notice after running the cell above.)*

## 4. Target variable and class balance
If one class dominates, accuracy alone can be misleading.

In [ ]:
target_col = "HeartDisease"
print(df[target_col].value_counts())
print()
print(df[target_col].value_counts(normalize=True).round(3))

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=target_col, data=df)
plt.title("Heart Disease Distribution")
plt.xlabel("HeartDisease (0 = no, 1 = yes)")
plt.ylabel("Count")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 5. Feature types

In [ ]:
numeric_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
binary_cols = ["FastingBS"]
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

print("Numeric:    ", numeric_cols)
print("Binary:     ", binary_cols)
print("Categorical:", categorical_cols)

Numeric columns are measured quantities; categorical columns hold text labels that must be **encoded** before modeling (done in the model notebooks).

## 6. Univariate analysis
### 6.1 Numeric features
Look at the shape of each histogram: symmetric, skewed, more than one peak?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(df[col], bins=20, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

### 6.2 Categorical features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), categorical_cols + binary_cols):
    sns.countplot(x=col, data=df, ax=ax)
    ax.set_title(f"Count of {col}")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 7. Bivariate analysis (features vs. target)
### 7.1 Numeric features vs. HeartDisease
Box plots: compare the median and spread of each feature for patients with and without heart disease.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=target_col, y=col, data=df, ax=ax)
    ax.set_title(f"{col} by HeartDisease")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

### 7.2 Categorical features vs. HeartDisease
This is a **grouped bar chart**: each category has one bar per class, side by side.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), categorical_cols + binary_cols):
    sns.countplot(x=col, hue=target_col, data=df, ax=ax)
    ax.set_title(f"{col} vs HeartDisease")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

Counts can be misleading when categories have different sizes, so we also compute the **percentage** of patients with heart disease inside each category.

In [ ]:
for col in categorical_cols + binary_cols:
    print(f"--- {col} ---")
    print(df.groupby(col)[target_col].mean().round(3).sort_values(ascending=False))
    print()

**Observation:**

*(Write what you notice after running the cell above.)*

### 7.3 Are the zero cholesterol values random?
If the zeros were random, they would be spread evenly over both classes. Compare below.

In [ ]:
df["Cholesterol_is_zero"] = (df["Cholesterol"] == 0).astype(int)
print(pd.crosstab(df["Cholesterol_is_zero"], df[target_col]))
print()
print(pd.crosstab(df["Cholesterol_is_zero"], df[target_col], normalize="index").round(3))
df = df.drop(columns="Cholesterol_is_zero")   # temporary helper column

**Observation:**

*(Write what you notice after running the cell above.)*

## 8. Correlation
Correlation measures the *linear* relationship between numeric variables (-1 to +1). The diagonal is always 1.

In [ ]:
corr_cols = numeric_cols + binary_cols + [target_col]
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## Summary of issues found (fill this in)

List the data-quality problems you found in this notebook. They will be fixed in **notebook 2 (preprocessing)**.

1. 
2. 
3. 

---
### Task
Use `groupby` to compute the mean `Age` and mean `MaxHR` for each value of `HeartDisease`, and explain the difference in one sentence.